In [ ]:
import os
import copy
import math
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split

import tonic
import tonic.transforms as transforms

from sinabs.layers import SumPool2d
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from pathlib import Path


# =========================
# Config
# =========================
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATA_ROOT = Path.home() / "Desktop" / "data"
OUT_DIR = Path.home() / "Desktop" / "outputs_dvsgesture_speck"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TIME_BINS = 20
BATCH_SIZE = 8
EPOCHS = 2
LR = 1e-3
WEIGHT_DECAY = 1e-5
VAL_SPLIT = 0.1
NUM_CLASSES = 11

CLASS_NAMES = [
    "Hand clapping",
    "Right hand wave",
    "Left hand wave",
    "Right arm cw",
    "Right arm ccw",
    "Left arm cw",
    "Left arm ccw",
    "Arm roll",
    "Air drums",
    "Air guitar",
    "Other",
]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# =========================
# Dataset
# =========================
frame_transform = transforms.Compose([
    transforms.ToFrame(sensor_size=(128, 128, 2), n_time_bins=TIME_BINS),
])

train_full = tonic.datasets.DVSGesture(
    save_to=str(DATA_ROOT),
    train=True,
    transform=frame_transform,
)

test_set = tonic.datasets.DVSGesture(
    save_to=str(DATA_ROOT),
    train=False,
    transform=frame_transform,
)

val_size = int(len(train_full) * VAL_SPLIT)
train_size = len(train_full) - val_size
train_set, val_set = random_split(
    train_full,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED),
)


def collate_fn(batch):
    frames, labels = zip(*batch)
    # Tonic ToFrame returns [T, C, H, W]
    x = np.stack(frames).astype(np.float32)
    # optional normalization / clipping
    x = np.clip(x, 0.0, 1.0)
    y = np.array(labels, dtype=np.int64)

    x = torch.from_numpy(x)   # [B, T, C, H, W]
    y = torch.from_numpy(y)
    return x, y


train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
    collate_fn=collate_fn,
)

test_loader = DataLoader(
    test_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
    collate_fn=collate_fn,
)


# =========================
# Model
# =========================
class GestureANN(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            # Core 0
            nn.Conv2d(2, 8, kernel_size=3, stride=2, padding=0, bias=False),   # 128 -> 63
            nn.ReLU(),
            SumPool2d(2, 2),                                                    # 63 -> 31

            # Core 1
            nn.Conv2d(8, 16, kernel_size=3, stride=1, padding=0, bias=False),  # 31 -> 29
            nn.ReLU(),
            SumPool2d(2, 2),                                                    # 29 -> 14

            # Core 2
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=0, bias=False), # 14 -> 12
            nn.ReLU(),
            SumPool2d(2, 2),                                                    # 12 -> 6

            # Core 3
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=0, bias=False), # 6 -> 4
            nn.ReLU(),

            # "FC" replacement layers
            nn.Conv2d(64, 64, kernel_size=4, stride=1, padding=0, bias=False), # 4 -> 1
            nn.ReLU(),

            nn.Conv2d(64, 128, kernel_size=1, bias=False),
            nn.ReLU(),

            nn.Conv2d(128, 256, kernel_size=1, bias=False),
            nn.ReLU(),

            nn.Conv2d(256, 11, kernel_size=1, bias=False),                      # -> (11,1,1)
        )

    def forward(self, x):
        """
        x: [B, T, C, H, W]
        We run each time bin through the same topology and sum logits across time.
        """
        b, t, c, h, w = x.shape
        x = x.view(b * t, c, h, w)
        y = self.net(x)                 # [B*T, 11, 1, 1]
        y = y.view(b, t, NUM_CLASSES)   # [B, T, 11]
        y = y.sum(dim=1)                # temporal aggregation
        return y


model = GestureANN().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)


# =========================
# Training / Eval helpers
# =========================
def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    all_preds = []
    all_targets = []

    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        logits = model(x)
        loss = criterion(logits, y)

        if training:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        preds = logits.argmax(dim=1)
        total_loss += loss.item() * y.size(0)
        total_correct += (preds == y).sum().item()
        total_samples += y.size(0)

        all_preds.append(preds.detach().cpu())
        all_targets.append(y.detach().cpu())

    avg_loss = total_loss / max(total_samples, 1)
    avg_acc = total_correct / max(total_samples, 1)

    all_preds = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()

    return avg_loss, avg_acc, all_targets, all_preds


# =========================
# Train
# =========================
history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
}

best_val_acc = -1.0
best_state = None

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc, _, _ = run_epoch(model, train_loader, optimizer=optimizer)
    val_loss, val_acc, _, _ = run_epoch(model, val_loader, optimizer=None)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())

# restore best model
model.load_state_dict(best_state)


# =========================
# Final evaluation
# =========================
train_loss, train_acc, _, _ = run_epoch(model, train_loader, optimizer=None)
val_loss, val_acc, _, _ = run_epoch(model, val_loader, optimizer=None)
test_loss, test_acc, y_true, y_pred = run_epoch(model, test_loader, optimizer=None)

print("\nFinal Results")
print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")
print(f"Test  Loss: {test_loss:.4f} | Test  Acc: {test_acc:.4f}")

cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)

plt.figure(figsize=(10, 10))
disp.plot(xticks_rotation=45, cmap="Blues", values_format="d")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "confusion_matrix.png"), dpi=300)
plt.close()

# Accuracy/Loss curves
epochs_axis = np.arange(1, EPOCHS + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs_axis, history["train_loss"], label="Train Loss")
plt.plot(epochs_axis, history["val_loss"], label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "loss_curve.png"), dpi=300)
plt.close()

plt.figure(figsize=(8, 5))
plt.plot(epochs_axis, history["train_acc"], label="Train Accuracy")
plt.plot(epochs_axis, history["val_acc"], label="Val Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "accuracy_curve.png"), dpi=300)
plt.close()

# Save metrics text
with open(os.path.join(OUT_DIR, "metrics.txt"), "w") as f:
    f.write(f"Train Loss: {train_loss:.6f}\n")
    f.write(f"Train Acc:  {train_acc:.6f}\n")
    f.write(f"Val Loss:   {val_loss:.6f}\n")
    f.write(f"Val Acc:    {val_acc:.6f}\n")
    f.write(f"Test Loss:  {test_loss:.6f}\n")
    f.write(f"Test Acc:   {test_acc:.6f}\n")

# Save full trainable model state
torch.save(model.state_dict(), os.path.join(OUT_DIR, "gesture_ann_full_state.pt"))

# Save only convolution weights, which are what your Speck SNN needs
conv_only_state = {
    k.replace("net.", "", 1): v
    for k, v in model.state_dict().items()
    if k.endswith(".weight")
}
torch.save(conv_only_state, OUT_DIR / "gesture_speck_conv_weights.pt")

print(f"\nSaved files to: {OUT_DIR}")
print("Main deployment file: gesture_speck_conv_weights.pt")

Epoch 01/2 | train_loss=1.4477 train_acc=0.4464 | val_loss=1.1744 val_acc=0.5140
Epoch 02/2 | train_loss=0.9023 train_acc=0.6351 | val_loss=0.9015 val_acc=0.5981

Final Results
Train Loss: 0.6713 | Train Acc: 0.6979
Val   Loss: 0.9015 | Val   Acc: 0.5981
Test  Loss: 0.8308 | Test  Acc: 0.6742

Saved files to: /Users/briccs/Desktop/outputs_dvsgesture_speck
Main deployment file: gesture_speck_conv_weights.pt


<Figure size 1000x1000 with 0 Axes>

In [5]:
frame_transform = transforms.Compose([
    transforms.ToFrame(sensor_size=(128, 128, 2), n_time_bins=TIME_BINS),
])

train_full = tonic.datasets.DVSGesture(
    save_to=str(DATA_ROOT),
    train=True,
    transform=frame_transform,
)

test_set = tonic.datasets.DVSGesture(
    save_to=str(DATA_ROOT),
    train=False,
    transform=frame_transform,
)

print("Train size:", len(train_full))
print("Test size:", len(test_set))

Train size: 1077
Test size: 264
